# Network architecture study (Tables 3-4)

Selects the network depth and width of the CA-PINN with the tanh activation:

1. Depth study (1 to 6 hidden layers) at a fixed width of 40 neurons.
2. Width study (10 to 100 neurons) at a fixed depth of 4 hidden layers.

Each architecture is trained with three random seeds and compared with one common finite-difference reference. This study used 5000 Adam epochs.

In [ ]:
####### Imports and project directory #######
#############################################

import os
import gc
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_DIR = Path(".")  # repository root (relative paths)

os.chdir(PROJECT_DIR)

PINN_NOTEBOOK = PROJECT_DIR / "CA_PINN_Solver.ipynb"
FD_NOTEBOOK = PROJECT_DIR / "FD_Solver.ipynb"
VALIDATION_FILE = PROJECT_DIR / "validation_utils.py"

for required_file in [
    PINN_NOTEBOOK,
    FD_NOTEBOOK,
    VALIDATION_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file not found: {required_file}"
        )

from validation_utils import rel_l2

print("Working directory:", Path.cwd())
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
######## Reproducibility settings #########
###########################################

os.environ["TF_DETERMINISTIC_OPS"] = "1"

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print(
        "Could not explicitly enable deterministic operations:",
        repr(exc),
    )

def set_all_seeds(seed: int) -> None:
    """Reset Python, NumPy, and TensorFlow random states."""
    tf.keras.backend.clear_session()
    gc.collect()

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [ ]:
# ============================================================
# EXECUTION SWITCHES
# ============================================================

### Study configuration ###
###########################

# Run the notebook first with TEST_MODE = True #
# This performs only one architecture and one seed. Once it works, change it to False #

TEST_MODE = False

# False: preserve completed runs and skip them.
# True: rerun and replace completed runs.
OVERWRITE_EXISTING = False

# Generate a new FD reference.
# Set this to False after the reference has been generated.
RUN_FD_REFERENCE = False

# Train the PINN cases.
RUN_PINN_CASES = False

# ============================================================
# BASELINE PHYSICAL PARAMETERS
# ============================================================

BASE_P0 = 1.0e21
BASE_PHI = 0.10

BASE_WCONS = 100.0

# ============================================================
# FIXED TRAINING PARAMETERS FOR ARCHITECTURE SELECTION
# ============================================================

BASE_ACTIVATION = "tanh"

NUM_EPOCH_ADAM = 5000
NUM_EPOCH_LBFGS = 10000
RUN_LBFGS = True

# ============================================================
# RANDOM SEEDS
# ============================================================

FULL_SEEDS = [42, 1234, 2024]

if TEST_MODE:
    SEEDS = [42]
else:
    SEEDS = FULL_SEEDS

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

OUTPUT_ROOT = (
    PROJECT_DIR
    / "results"
    / "architecture"
)

FD_REFERENCE_DIR = OUTPUT_ROOT / "fd_reference"
RUNS_DIR = OUTPUT_ROOT / "runs"
FIGURES_DIR = OUTPUT_ROOT / "figures"

for directory in [
    OUTPUT_ROOT,
    FD_REFERENCE_DIR,
    RUNS_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RUNS_CSV = OUTPUT_ROOT / "architecture_all_runs.csv"
SUMMARY_CSV = OUTPUT_ROOT / "architecture_mean_std.csv"

print("Test mode:", TEST_MODE)
print("Seeds:", SEEDS)
print("Output root:", OUTPUT_ROOT)

In [ ]:
# ============================================================
# DEPTH STUDY
# Width is fixed at 40 neurons per hidden layer.
# ============================================================

DEPTH_CASES = [
    (1, 40),
    (2, 40),
    (3, 40),
    (4, 40),
    (5, 40),
    (6, 40),
]

# ============================================================
# WIDTH STUDY
# Depth is fixed at 4 hidden layers.
# ============================================================

WIDTH_CASES = [
    (4, 10),
    (4, 20),
    (4, 40),
    (4, 60),
    (4, 80),
    (4, 100),
]

# Remove the duplicate 4x40 configuration.
ARCHITECTURE_CASES = sorted(
    set(DEPTH_CASES + WIDTH_CASES)
)

if TEST_MODE:
    ARCHITECTURE_CASES = [(4, 40)]

print("Unique architectures:")
for hidden_layers, neurons in ARCHITECTURE_CASES:
    print(f"  {hidden_layers} x {neurons}")

print(
    "\nNumber of PINN runs:",
    len(ARCHITECTURE_CASES) * len(SEEDS),
)

In [ ]:
##### Helper functions #####
############################

REQUIRED_PINN_FILES = [
    "pinn_psi.csv",
    "pinn_n.csv",
    "pinn_I.csv",
    "loss_history.csv",
]

REQUIRED_FD_FILES = [
    "fd_psi.csv",
    "fd_n.csv",
    "fd_I.csv",
]

LEFTOVER_GLOBALS = [
    "prebias_model",
    "main_model",
    "model",
    "layers",
    "results",
    "X_0",
    "X_r",
    "X_a",
    "X_c",
    "u_a",
    "u_c",
    "X_r_pre",
    "X_0_pre",
    "X_aPre",
    "X_cPre",
    "lb",
    "ub",
    "lb_pre",
    "ub_pre",
    "lb_main",
    "ub_main",
    "u0_interp_dict",
    "u_0",
    "x_0",
    "X_test",
    "X_list",
    "T_list",
    "psi_pred",
    "n_pred",
    "I_pred",
]

def clear_solver_globals() -> None:
    """Remove variables left by previous notebook executions."""
    namespace = globals()

    for name in LEFTOVER_GLOBALS:
        namespace.pop(name, None)

    tf.keras.backend.clear_session()
    gc.collect()

def require_files(
    directory: Path,
    filenames: list[str],
) -> None:
    """Raise an error if expected outputs are missing."""
    missing = [
        filename
        for filename in filenames
        if not (directory / filename).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"Missing files in {directory}: {missing}"
        )

def copy_fd_reference(case_dir: Path) -> None:
    """Copy the common FD reference into one PINN case folder."""
    require_files(FD_REFERENCE_DIR, REQUIRED_FD_FILES)

    for filename in REQUIRED_FD_FILES:
        shutil.copy2(
            FD_REFERENCE_DIR / filename,
            case_dir / filename,
        )

def compute_case_metrics(case_dir: Path) -> dict:
    """Compute validation metrics without generating a plot."""
    require_files(
        case_dir,
        REQUIRED_PINN_FILES + REQUIRED_FD_FILES,
    )

    pinn_psi = pd.read_csv(case_dir / "pinn_psi.csv")
    pinn_n = pd.read_csv(case_dir / "pinn_n.csv")
    pinn_I = pd.read_csv(case_dir / "pinn_I.csv")

    fd_psi = pd.read_csv(case_dir / "fd_psi.csv")
    fd_n = pd.read_csv(case_dir / "fd_n.csv")
    fd_I = pd.read_csv(case_dir / "fd_I.csv")

    # The PINN and FD files should already use the same grids.
    for name, pinn_df, fd_df in [
        ("psi", pinn_psi, fd_psi),
        ("n", pinn_n, fd_n),
        ("I", pinn_I, fd_I),
    ]:
        if len(pinn_df) != len(fd_df):
            raise ValueError(
                f"{name}: PINN and FD grid lengths differ."
            )

        max_dx = np.max(
            np.abs(
                pinn_df["x_nm"].to_numpy()
                - fd_df["x_nm"].to_numpy()
            )
        )

        if max_dx > 1.0e-9:
            raise ValueError(
                f"{name}: PINN and FD grids differ. "
                f"Maximum difference = {max_dx:.3e} nm."
            )

    I_mean_pinn = float(pinn_I["I_m3"].mean())
    I_mean_fd = float(fd_I["I_m3"].mean())

    metrics = {
        "relL2_psi": float(
            rel_l2(
                pinn_psi["psi_V"],
                fd_psi["psi_V"],
            )
        ),
        "relL2_n": float(
            rel_l2(
                pinn_n["n_m3"],
                fd_n["n_m3"],
            )
        ),
        "relL2_I": float(
            rel_l2(
                pinn_I["I_m3"],
                fd_I["I_m3"],
            )
        ),
        "I_mean_PINN": I_mean_pinn,
        "I_mean_FD": I_mean_fd,
        "I_mean_rel_error": (
            abs(I_mean_pinn - I_mean_fd)
            / (abs(I_mean_fd) + 1.0e-300)
        ),
    }

    # Geometric mean of the three field errors.
    metrics["aggregate_error"] = float(
        np.exp(
            np.mean(
                np.log(
                    np.maximum(
                        [
                            metrics["relL2_psi"],
                            metrics["relL2_n"],
                            metrics["relL2_I"],
                        ],
                        1.0e-300,
                    )
                )
            )
        )
    )

    loss_df = pd.read_csv(
        case_dir / "loss_history.csv"
    )

    if "total_loss" not in loss_df.columns:
        raise ValueError(
            "The loss-history file does not contain "
            "a 'total_loss' column."
        )

    metrics["final_total_loss"] = float(
        loss_df["total_loss"].iloc[-1]
    )

    return metrics

def count_trainable_parameters() -> int:
    """Count parameters in the trained PINN."""
    if "prebias_model" not in globals():
        raise RuntimeError(
            "prebias_model is unavailable after PINN execution."
        )

    variables = prebias_model.model.trainable_variables

    return int(
        sum(
            np.prod(variable.shape)
            for variable in variables
        )
    )

In [ ]:
### Generate the finite-difference reference once ###
#####################################################

if RUN_FD_REFERENCE:
    print("=" * 80)
    print("Generating the common FD reference")
    print("=" * 80)

    clear_solver_globals()

    globals()["P0"] = BASE_P0
    globals()["phia"] = BASE_PHI
    globals()["phic"] = BASE_PHI
    globals()["OUTPUT_DIR"] = str(FD_REFERENCE_DIR)

    fd_start = time.time()

    get_ipython().run_line_magic(
        "run",
        f'-i "{FD_NOTEBOOK}"',
    )

    fd_runtime = time.time() - fd_start

    require_files(
        FD_REFERENCE_DIR,
        REQUIRED_FD_FILES,
    )

    print(
        f"FD reference completed in "
        f"{fd_runtime:.1f} s."
    )

else:
    require_files(
        FD_REFERENCE_DIR,
        REQUIRED_FD_FILES,
    )

    print(
        "Using the existing FD reference in:",
        FD_REFERENCE_DIR,
    )

In [ ]:
def recover_existing_run_metadata(
    hidden_layers: int,
    neurons: int,
    seed: int,
):
    """
    Recover runtime and trainable-parameter metadata
    from the existing architecture CSV.
    """

    csv_file = RUNS_DIR.parent / "architecture_all_runs.csv"

    if not csv_file.exists():
        return np.nan, np.nan

    try:
        old_df = pd.read_csv(csv_file)

        required_columns = {
            "hidden_layers",
            "neurons",
            "seed",
            "training_runtime_s",
            "trainable_parameters",
        }

        if not required_columns.issubset(old_df.columns):
            return np.nan, np.nan

        match = old_df[
            (old_df["hidden_layers"] == hidden_layers)
            & (old_df["neurons"] == neurons)
            & (old_df["seed"] == seed)
        ]

        if match.empty:
            return np.nan, np.nan

        runtime = match.iloc[-1]["training_runtime_s"]
        parameters = match.iloc[-1]["trainable_parameters"]

        return runtime, parameters

    except Exception as exc:
        print(
            "Warning: could not recover existing metadata:",
            repr(exc),
        )
        return np.nan, np.nan

In [ ]:
### Function for one architecture and seed ###
##############################################

def run_architecture_case(
    hidden_layers: int,
    neurons: int,
    seed: int,
) -> dict:
    """Train and validate one architecture for one seed."""

    architecture = f"{hidden_layers}x{neurons}"

    tag = (
        f"L{hidden_layers}_W{neurons}"
        f"_tanh_seed{seed}"
    )

    case_dir = RUNS_DIR / tag
    case_dir.mkdir(parents=True, exist_ok=True)

    completed_files = (
        REQUIRED_PINN_FILES
        + REQUIRED_FD_FILES
    )

    already_complete = all(
        (case_dir / filename).exists()
        for filename in completed_files
    )

    if already_complete and not OVERWRITE_EXISTING:
        print(
            f"\nSkipping completed case: {tag}"
        )

        metrics = compute_case_metrics(case_dir)

        existing_runtime, existing_parameters = (
            recover_existing_run_metadata(
                hidden_layers=hidden_layers,
                neurons=neurons,
                seed=seed,
            )
        )

        return {
            "architecture": architecture,
            "hidden_layers": hidden_layers,
            "neurons": neurons,
            "seed": seed,
            "activation": BASE_ACTIVATION,
            "P0": BASE_P0,
            "phi": BASE_PHI,
            "W_cons": BASE_WCONS,
            "num_epoch_adam": NUM_EPOCH_ADAM,
            "num_epoch_lbfgs": NUM_EPOCH_LBFGS,
            "run_lbfgs": RUN_LBFGS,
            "training_runtime_s": existing_runtime,
            "trainable_parameters": existing_parameters,
            **metrics,
            "status": "reused",
            "case_dir": str(case_dir),
        }

    print("\n" + "=" * 80)
    print(f"Running architecture case: {tag}")
    print(f"Architecture: {architecture}")
    print(f"Seed: {seed}")
    print(f"Activation: {BASE_ACTIVATION}")
    print(f"Output: {case_dir}")
    print("=" * 80)

    clear_solver_globals()
    set_all_seeds(seed)

    # Parameters consumed by CA_PINN_Solver.ipynb
    globals()["P0"] = BASE_P0
    globals()["phia"] = BASE_PHI
    globals()["phic"] = BASE_PHI

    globals()["HIDDEN_LAYERS"] = hidden_layers
    globals()["NEURONS"] = neurons
    globals()["ACTIVATION"] = BASE_ACTIVATION

    globals()["W_cons"] = BASE_WCONS

    globals()["NUM_EPOCH_ADAM"] = NUM_EPOCH_ADAM
    globals()["NUM_EPOCH_LBFGS"] = NUM_EPOCH_LBFGS
    globals()["RUN_LBFGS"] = RUN_LBFGS

    globals()["OUTPUT_DIR"] = str(case_dir)

    training_start = time.time()

    try:
        if RUN_PINN_CASES:
            get_ipython().run_line_magic(
                "run",
                f'-i "{PINN_NOTEBOOK}"',
            )

        training_runtime = (
            time.time() - training_start
        )

        require_files(
            case_dir,
            REQUIRED_PINN_FILES,
        )

        trainable_parameters = (
            count_trainable_parameters()
        )

        # Every architecture uses the same FD reference.
        copy_fd_reference(case_dir)

        metrics = compute_case_metrics(case_dir)

        row = {
            "architecture": architecture,
            "hidden_layers": hidden_layers,
            "neurons": neurons,
            "seed": seed,
            "activation": BASE_ACTIVATION,
            "P0": BASE_P0,
            "phi": BASE_PHI,
            "W_cons": BASE_WCONS,
            "num_epoch_adam": NUM_EPOCH_ADAM,
            "num_epoch_lbfgs": NUM_EPOCH_LBFGS,
            "run_lbfgs": RUN_LBFGS,
            "training_runtime_s": float(
                training_runtime
            ),
            "trainable_parameters": (
                trainable_parameters
            ),
            **metrics,
            "status": "ok",
            "case_dir": str(case_dir),
        }

        print(
            f"[{tag}] "
            f"aggregate error = "
            f"{metrics['aggregate_error']:.3e}"
        )

        return row

    except Exception as exc:
        print(
            f"[{tag}] FAILED: {repr(exc)}"
        )

        return {
            "architecture": architecture,
            "hidden_layers": hidden_layers,
            "neurons": neurons,
            "seed": seed,
            "activation": BASE_ACTIVATION,
            "P0": BASE_P0,
            "phi": BASE_PHI,
            "W_cons": BASE_WCONS,
            "num_epoch_adam": NUM_EPOCH_ADAM,
            "num_epoch_lbfgs": NUM_EPOCH_LBFGS,
            "run_lbfgs": RUN_LBFGS,
            "training_runtime_s": float(
                time.time() - training_start
            ),
            "trainable_parameters": np.nan,
            "relL2_psi": np.nan,
            "relL2_n": np.nan,
            "relL2_I": np.nan,
            "I_mean_PINN": np.nan,
            "I_mean_FD": np.nan,
            "I_mean_rel_error": np.nan,
            "aggregate_error": np.nan,
            "final_total_loss": np.nan,
            "status": (
                f"failed: {type(exc).__name__}"
            ),
            "case_dir": str(case_dir),
        }

In [ ]:
### Run the architecture sweep ###
##################################

# Preserve completed rows if the notebook is resumed.
if RUNS_CSV.exists() and not OVERWRITE_EXISTING:
    existing_df = pd.read_csv(RUNS_CSV)
    existing_rows = existing_df.to_dict("records")
else:
    existing_rows = []

existing_keys = {
    (
        int(row["hidden_layers"]),
        int(row["neurons"]),
        int(row["seed"]),
    )
    for row in existing_rows
    if str(row.get("status", "")).startswith(
        ("ok", "reused")
    )
}

rows = existing_rows.copy()

for hidden_layers, neurons in ARCHITECTURE_CASES:
    for seed in SEEDS:
        key = (
            hidden_layers,
            neurons,
            seed,
        )

        if (
            key in existing_keys
            and not OVERWRITE_EXISTING
        ):
            print(
                "Already recorded, skipping:",
                key,
            )
            continue

        row = run_architecture_case(
            hidden_layers=hidden_layers,
            neurons=neurons,
            seed=seed,
        )

        rows.append(row)

        pd.DataFrame(rows).to_csv(
            RUNS_CSV,
            index=False,
        )

        print(
            "Updated:",
            RUNS_CSV,
        )

print("\nArchitecture sweep completed.")

In [ ]:
### Inspect all individual runs ###
###################################

runs_df = pd.read_csv(RUNS_CSV)

display_columns = [
    "architecture",
    "seed",
    "relL2_psi",
    "relL2_n",
    "relL2_I",
    "aggregate_error",
    "I_mean_rel_error",
    "training_runtime_s",
    "trainable_parameters",
    "status",
]

display(
    runs_df
    .sort_values(
        ["hidden_layers", "neurons", "seed"]
    )[display_columns]
    .reset_index(drop=True)
)

In [ ]:
### Aggregate mean and standard deviation ###
#############################################

successful_df = runs_df[
    runs_df["status"].astype(str).str.startswith(
        ("ok", "reused")
    )
].copy()

metric_columns = [
    "relL2_psi",
    "relL2_n",
    "relL2_I",
    "aggregate_error",
    "I_mean_rel_error",
    "final_total_loss",
    "training_runtime_s",
    "trainable_parameters",
]

summary_df = (
    successful_df
    .groupby(
        [
            "hidden_layers",
            "neurons",
            "architecture",
        ],
        as_index=False,
    )[metric_columns]
    .agg(["mean", "std", "count"])
)

# Flatten the multi-level column names.
summary_df.columns = [
    "_".join(
        str(value)
        for value in column
        if str(value)
    ).rstrip("_")
    if isinstance(column, tuple)
    else column
    for column in summary_df.columns
]

# Restore clear identifier names after flattening.
summary_df = summary_df.rename(
    columns={
        "hidden_layers": "hidden_layers",
        "neurons": "neurons",
        "architecture": "architecture",
    }
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

print("Saved:", SUMMARY_CSV)

display(
    summary_df.sort_values(
        "aggregate_error_mean"
    ).reset_index(drop=True)
)

# Depth study: width fixed at 40 neurons
depth_df = (
    summary_df[
        summary_df["neurons"] == 40
    ]
    .sort_values("hidden_layers")
    .reset_index(drop=True)
)

# Width study: depth fixed at 3 hidden layers
width_df = (
    summary_df[
        summary_df["hidden_layers"] == 4
    ]
    .sort_values("neurons")
    .reset_index(drop=True)
)

print("Depth cases:", len(depth_df))
print("Width cases:", len(width_df))

In [ ]:
### Separate depth and width results ###
########################################

grouped = successful_df.groupby(
    [
        "hidden_layers",
        "neurons",
        "architecture",
    ]
)

summary_rows = []

for identifiers, group in grouped:
    hidden_layers, neurons, architecture = identifiers

    row = {
        "hidden_layers": hidden_layers,
        "neurons": neurons,
        "architecture": architecture,
        "num_successful_seeds": len(group),
    }

    for metric in metric_columns:
        values = group[metric].dropna()

        row[f"{metric}_mean"] = (
            float(values.mean())
            if len(values)
            else np.nan
        )

        row[f"{metric}_std"] = (
            float(values.std(ddof=1))
            if len(values) > 1
            else np.nan
        )

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

display(
    summary_df.sort_values(
        "aggregate_error_mean"
    ).reset_index(drop=True)
)